# LayerLog Anatomy

Looks at aggregate per-layer records and how pass-qualified labels relate to no-pass labels.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerPassLog.layer_total_num",
    "tl.LayerPassLog.layer_type",
    "tl.LayerPassLog.layer_type_num",
    "tl.LayerPassLog.leaf_module_pass",
    "tl.LayerPassLog.log_tensor_grad",
    "tl.LayerPassLog.lookup_keys",
    "tl.LayerPassLog.macs_backward",
    "tl.LayerPassLog.macs_forward",
    "tl.LayerPassLog.materialize_activation",
    "tl.LayerPassLog.materialize_gradient",
    "tl.LayerPassLog.max_distance_from_input",
    "tl.LayerPassLog.max_distance_from_output",
    "tl.LayerPassLog.min_distance_from_input",
    "tl.LayerPassLog.min_distance_from_output",
    "tl.LayerPassLog.module_address_normalized",
    "tl.LayerPassLog.module_entry_exit_thread_output",
    "tl.LayerPassLog.module_entry_exit_threads_inputs",
    "tl.LayerPassLog.module_nesting_depth",
    "tl.LayerPassLog.module_passes_entered",
    "tl.LayerPassLog.module_passes_exited",
    "tl.LayerPassLog.modules_entered",
    "tl.LayerPassLog.modules_entered_argnames",
    "tl.LayerPassLog.modules_exited",
    "tl.LayerPassLog.num_args",
    "tl.LayerPassLog.num_keyword_args",
    "tl.LayerPassLog.num_param_tensors",
    "tl.LayerPassLog.num_params_frozen",
    "tl.LayerPassLog.num_params_total",
    "tl.LayerPassLog.num_params_trainable",
    "tl.LayerPassLog.num_passes",
    "tl.LayerPassLog.num_positional_args",
    "tl.LayerPassLog.operation_equivalence_type",
    "tl.LayerPassLog.operation_num",
    "tl.LayerPassLog.output_descendants",
    "tl.LayerPassLog.output_device",
    "tl.LayerPassLog.output_path",
    "tl.LayerPassLog.params",
    "tl.LayerPassLog.params_memory",
    "tl.LayerPassLog.params_memory_str",
    "tl.LayerPassLog.parent_layer_arg_locs",
    "tl.LayerPassLog.parent_layers",
    "tl.LayerPassLog.parent_param_barcodes",
    "tl.LayerPassLog.parent_param_logs",
    "tl.LayerPassLog.parent_param_passes",
    "tl.LayerPassLog.parent_param_shapes",
    "tl.LayerPassLog.parent_params",
    "tl.LayerPassLog.pass_num",
    "tl.LayerPassLog.passes",
    "tl.LayerPassLog.recurrent_group",
    "tl.LayerPassLog.root_ancestors",
    "tl.LayerPassLog.save_gradients",
    "tl.LayerPassLog.save_tensor_data",
    "tl.LayerPassLog.scalar_bool_value",
    "tl.LayerPassLog.show",
    "tl.LayerPassLog.sibling_layers",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")